# Student Information Tool (SJIT) — LangChain Agent with Gemini

This notebook builds a **LangChain Agent** powered by **Gemini** that answers
student-related questions by *deciding for itself* which tools to call and in
what order.

**Database:** a local SQLite database (`students.db`) holding student marks.

**Tools the agent can choose from:**

| Tool | Purpose |
|---|---|
| `get_student_info(student_id)` | Name + Department |
| `get_student_marks(student_id)` | Python / Database / AI / Web marks |
| `calculator(expression)` | Total / average marks (safe arithmetic) |
| `get_passing_rules()` | University passing rules |

**Key idea — agentic vs. fixed chain:** we never hard-code
"call A, then B, then C". We hand the LLM all four tools and a question, and
the **LLM itself** decides which tool(s) it needs, calls them one at a time,
looks at the results, and decides whether it needs another tool — repeating
until it can produce a Final Answer. This is the classic

```
Question → LLM → pick tool → run tool → LLM → need another tool? → ... → Final Answer
```

loop, implemented here with LangChain's `create_agent` (the current,
`langgraph`-based agent constructor in `langchain` 1.x).

> **Note on LangChain versions:** LangChain's agent API changed in the 1.x
> release line. The older `create_tool_calling_agent` + `AgentExecutor`
> classes have been removed from `langchain.agents`. This notebook targets
> `langchain>=1.0`, which exposes a single `create_agent(model, tools, ...)`
> function that returns a ready-to-run agent (internally a compiled
> LangGraph graph). If you're on an older `langchain` (0.x), you'd use
> `create_tool_calling_agent`/`AgentExecutor` instead — but this notebook
> installs and uses the current API.


## Step 1 — Install dependencies

We need:
- `langchain` / `langchain-core` — the agent framework
- `langchain-google-genai` — the Gemini chat model wrapper
- `google-generativeai` — underlying Gemini SDK

Run this cell once (uncomment if running fresh).


In [ ]:
!pip install -U "langchain>=1.0" langchain-core langchain-google-genai --quiet


## Step 2 — Configure your Gemini API key

Get a free key from https://aistudio.google.com/app/apikey and either:

- set it as an environment variable `GOOGLE_API_KEY` before starting Jupyter, **or**
- paste it into the `getpass` prompt below (nothing is echoed to the screen, and
  it is *not* saved into the notebook file).


In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY_HERE"  # <-- paste your Gemini API key here

print("API key configured:", bool(os.environ.get("GOOGLE_API_KEY")))


## Step 3 — Create the SQLite database

We create `students.db` with a single `students` table and load the five
rows given in the problem statement. Re-running this cell drops and
recreates the table, so it's safe to re-run.


In [ ]:
import sqlite3

DB_PATH = "students.db"

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS students")
cur.execute('''
    CREATE TABLE students (
        student_id TEXT PRIMARY KEY,
        name       TEXT NOT NULL,
        department TEXT NOT NULL,
        python     INTEGER NOT NULL,
        database   INTEGER NOT NULL,
        ai         INTEGER NOT NULL,
        web        INTEGER NOT NULL
    )
''')

students_data = [
    ("22CS045", "Dhanushya", "Computer Science",       85, 72, 90, 78),
    ("22CS046", "Rahul",     "Computer Science",       65, 70, 68, 72),
    ("22CS047", "Priya",     "Information Technology", 92, 88, 95, 90),
    ("22CS048", "Arun",      "Information Technology", 55, 60, 58, 62),
    ("22CS049", "Meena",     "Computer Science",       78, 85, 80, 88),
]

cur.executemany(
    "INSERT INTO students VALUES (?, ?, ?, ?, ?, ?, ?)",
    students_data
)

conn.commit()

# sanity check
for row in cur.execute("SELECT * FROM students"):
    print(row)

conn.close()


## Step 4 — Define the tools

Each tool is a plain Python function decorated with `@tool`. LangChain reads:

- the **function name** → tool name the LLM sees
- the **type hints** → the tool's input schema (so the LLM knows what
  arguments to pass, and in what type)
- the **docstring** → the tool description, which is what the LLM actually
  reads to decide *when* to use this tool

Good docstrings are what make automatic tool selection work well — they are
effectively "instructions to the LLM", not just human documentation.


In [ ]:
import sqlite3
from langchain_core.tools import tool


def _get_connection():
    return sqlite3.connect(DB_PATH)


@tool
def get_student_info(student_id: str) -> str:
    """Look up a student's NAME and DEPARTMENT given their student_id.

    Use this tool whenever the question asks who a student is, or asks for
    their name and/or department. Does NOT return marks.

    Args:
        student_id: The student's ID, e.g. "22CS045".
    """
    conn = _get_connection()
    cur = conn.cursor()
    cur.execute(
        "SELECT name, department FROM students WHERE student_id = ?",
        (student_id,),
    )
    row = cur.fetchone()
    conn.close()

    if row is None:
        return f"No student found with ID {student_id}."

    name, department = row
    return f"Student {student_id}: name={name}, department={department}"


@tool
def get_student_marks(student_id: str) -> str:
    """Look up a student's marks (Python, Database, AI, Web) given their student_id.

    Use this tool whenever the question involves marks, scores, totals,
    averages, or passing/failing — you must fetch the raw marks with this
    tool before you can compute anything about them.

    Args:
        student_id: The student's ID, e.g. "22CS045".
    """
    conn = _get_connection()
    cur = conn.cursor()
    cur.execute(
        "SELECT python, database, ai, web FROM students WHERE student_id = ?",
        (student_id,),
    )
    row = cur.fetchone()
    conn.close()

    if row is None:
        return f"No student found with ID {student_id}."

    python_mark, database_mark, ai_mark, web_mark = row
    return (
        f"Student {student_id} marks: "
        f"python={python_mark}, database={database_mark}, "
        f"ai={ai_mark}, web={web_mark}"
    )


@tool
def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression and return the numeric result.

    Use this tool any time you need to add up marks for a TOTAL, or divide
    to get an AVERAGE, or do any other arithmetic. Only give it a plain
    math expression, e.g. "85 + 72 + 90 + 78" or "(85+72+90+78)/4".
    Do not do the arithmetic yourself — always call this tool.

    Args:
        expression: A math expression using only numbers and + - * / ( ) . and spaces.
    """
    import re

    if not re.fullmatch(r"[0-9+\-*/(). %]+", expression):
        return "Error: expression contains disallowed characters."

    try:
        result = eval(expression, {"__builtins__": {}}, {})
    except Exception as exc:
        return f"Error evaluating expression: {exc}"

    return f"{expression} = {result}"


@tool
def get_passing_rules() -> str:
    """Return the university's passing rules.

    Use this tool whenever the question asks whether a student passes,
    is eligible, meets requirements, or anything about pass/fail criteria.
    Takes no arguments.
    """
    return (
        "University passing rules: "
        "1) Minimum overall average across all subjects must be >= 40%. "
        "2) Minimum mark in EACH individual subject must be >= 35%."
    )


tools = [get_student_info, get_student_marks, calculator, get_passing_rules]
print("Tools ready:", [t.name for t in tools])


## Step 5 — Set up the Gemini LLM

`temperature=0` keeps tool-selection deterministic and reliable, which is
what we want for this kind of structured, tool-driven task.



In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
)

print(llm.invoke("Reply with just the word 'ready' if you can hear me.").content)


## Step 6 — Build the Agent (not a fixed chain!)

This is the important part. We do **not** write:

```python
info = get_student_info(...)
marks = get_student_marks(...)
total = calculator(...)
```

Instead we give the LLM a system prompt and the list of tools, and let
`create_agent` build the loop for us:

```
LLM reads question
  → decides if a tool is needed, and which one
  → tool runs, result goes back to the LLM
  → LLM decides: do I need another tool, or can I answer now?
  → repeats until it emits a Final Answer
```

`create_agent(model, tools, system_prompt=...)` (from `langchain.agents`,
`langchain` 1.x) returns a compiled, runnable agent graph. Internally, each
turn it calls the model, checks whether the model's response contains tool
calls, runs any it finds, feeds the results back in as messages, and loops
until the model responds with no more tool calls — exactly the agentic loop
described above, with no fixed sequence written by us.


In [ ]:
from langchain.agents import create_agent

system_prompt = (
    "You are an assistant for a university Student Information system. "
    "You have access to tools for looking up a student's info, looking up "
    "their marks, doing arithmetic, and looking up passing rules. "
    "Never guess or make up marks, names, departments, or rules — always "
    "call the appropriate tool to get real data. When a question needs "
    "arithmetic (totals, averages, comparisons against a threshold), use "
    "the calculator tool rather than computing it yourself. "
    "Only call the tools that are actually needed for the question asked. "
    "Once you have everything you need, give a clear, complete final answer."
)

agent_executor = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
)

print("Agent ready.")


## Step 7 — A helper to run questions and show the tool-selection trace

`create_agent` returns a graph whose state is a list of chat `messages`.
We invoke it with `{"messages": [{"role": "user", "content": question}]}`
and get back the full message history: the user question, every
`AIMessage` (including any tool calls the model decided to make), every
`ToolMessage` (the tool's result), and the final `AIMessage` with no more
tool calls — that last one is the answer.

This small helper walks that history and prints each tool call as it
happened, so you can watch — in place of the old `verbose=True` — exactly
which tool(s) the LLM chose, in which order, purely from its own reasoning.


In [ ]:
def ask_agent(question: str) -> str:
    print("=" * 80)
    print("QUESTION:", question)
    print("=" * 80)

    result = agent_executor.invoke(
        {"messages": [{"role": "user", "content": question}]}
    )

    for msg in result["messages"]:
        tool_calls = getattr(msg, "tool_calls", None)
        if tool_calls:
            for call in tool_calls:
                print(f"  -> tool call: {call['name']}({call['args']})")
        elif msg.type == "tool":
            tool_name = getattr(msg, "name", "tool")
            print(f"  <- {tool_name} result: {msg.content}")

    final_answer = result["messages"][-1].content
    print("-" * 80)
    print("FINAL ANSWER:", final_answer)
    print()
    return final_answer


## Step 8 — Try the four sample questions

Watch the trace printed for each: it shows exactly which tool(s) the LLM
chose to call, and in what order — that choice is being made by the model,
not by our code.


**Question 1** — expects only `get_student_info`

In [ ]:
_ = ask_agent("What is the name and department of student 22CS045?")


**Question 2** — expects only `get_student_marks`

In [ ]:
_ = ask_agent("What are the marks of 22CS047?")


**Question 3** — expects `get_student_marks` → `calculator`

In [ ]:
_ = ask_agent("What is the total and average mark of 22CS045?")


**Question 4** — expects `get_student_marks` → `get_passing_rules` → `calculator`

In [ ]:
_ = ask_agent("Is 22CS045 eligible to pass according to the university rules?")


## Step 9 — The challenge question

Everything at once: identity, marks, arithmetic, and a pass/fail check.
The agent should chain **all four tools** on its own:

```
get_student_info() → get_student_marks() → calculator() → get_passing_rules() → Final Answer
```

(The exact order the LLM picks may vary slightly run to run — e.g. it might
fetch the passing rules before or after computing the average — since that
ordering is the model's own decision, not something we scripted. What matters
is that it correctly determines *all four* tools are needed.)


In [ ]:
challenge_question = (
    "I am 22CS045. Tell me my name, department, total marks, average marks, "
    "and whether I satisfy the university passing requirements."
)

_ = ask_agent(challenge_question)


## Recap — what this demonstrates

- **`@tool` + type hints + docstrings**: turns a normal Python function into
  something an LLM can discover, understand, and call with the right
  arguments — the docstring *is* the tool description the model reads.
- **No hard-coded sequence**: we never wrote `info(); marks(); calc()`. We
  only wired up the tool list; the LLM inspected the question each time and
  decided which tool(s) it needed and in what order.
- **Multi-hop tool use**: for Question 3, 4 and the challenge question, the
  agent's *first* tool call result (e.g. raw marks) became the *input* to a
  later tool call (e.g. the calculator expression) — this is the
  "one tool's result leads to another tool call" behavior from the
  learning objectives.
- **Agent loop vs. fixed chain**: `create_agent` implements the
  observe → think → act loop (ReAct-style, via tool calling) automatically;
  changing the question changes which tools fire, with zero code changes.
